<!-- NB_DOC_INTRO_V1 -->
# MLOps — Servir des modeles en production

> 📚 **Doc thematique** : [docs/08_MLOPS.md](docs/08_MLOPS.md) (MLOps)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Servir un modele ML en production** : transformer un pickle en service HTTP/gRPC scalable. Ce notebook compare les frameworks 2024-2025 (BentoML, TorchServe, Triton, vLLM, Ray Serve, KServe) avec **demo executable de l'approche FastAPI minimaliste** (la plus accessible), puis presente les autres en code de reference.

**Patterns de serving** : Online (sync REST), Online streaming (gRPC/WebSocket), Batch (Spark/Airflow), Embedded (mobile/edge). Cover aussi les **patterns prod critiques** : dynamic batching, A/B canary, caching, health checks.

Versions : `fastapi >= 0.110`, `pydantic >= 2.6`, `bentoml >= 1.2`.

## Plan

1. Patterns de serving (online / batch / streaming / edge)
2. Comparatif frameworks (BentoML, TorchServe, Triton, Ray Serve, vLLM, KServe)
3. **Demo executable** : FastAPI + sklearn (testable in-process)
4. **BentoML** — pattern recommande (code de reference)
5. **TorchServe** — PyTorch native
6. **Triton Inference Server** (NVIDIA) — multi-framework GPU
7. **Ray Serve** — Python distributed
8. **vLLM** — LLM-specific (PagedAttention)
9. Patterns prod : dynamic batching, A/B, canary, shadow, caching, health checks
10. Pieges et anti-patterns
11. References


## 1. Patterns de serving

| Pattern | Latence cible | Throughput | Cas d'usage |
|---|---|---|---|
| **Online synchrone** | < 100ms | Moyen | API REST, app temps reel |
| **Online streaming** | < 1s | Variable | Kafka, recommendation events |
| **Batch (asynchrone)** | minutes-heures | Tres haut | Daily scoring, ETL |
| **Embedded / Edge** | < 10ms | N/A | Mobile, IoT, navigateur (TFLite, ONNX, ExecuTorch) |
| **Inference-as-a-Service** | < 100ms | Gere | OpenAI, Replicate, HuggingFace Inference |

## 2. Comparatif frameworks 2024-2025

| Tool | Type | Forces | Limites |
|---|---|---|---|
| **FastAPI + uvicorn** | Custom | Flexible, simple, controle total | Tu codes tout (batching, monitoring, etc.) |
| **BentoML** | Framework | Packaging propre, K8s ready, multi-framework | Courbe d'apprentissage |
| **TorchServe** | PyTorch native | Tres optimise PyTorch, dynamic batching | Pas multi-framework |
| **Triton Inference Server** (NVIDIA) | Multi-framework | GPU-optimized, dynamic batching, ensemble | Setup lourd |
| **Ray Serve** | Distributed | Scale, multi-model, Python natif | Stack Ray a connaitre |
| **KServe** (ex-KFServing) | K8s native | Standard cloud-native ML | Demande K8s |
| **Modal / Replicate / Banana** | SaaS | Zero-ops, auto-scale | Vendor lock-in, cout |
| **vLLM** | LLM-specific | Throughput LLM ++++ (PagedAttention) | Que LLM |
| **MLflow serve** | Quick | Lie a MLflow | Pas prod-grade pour gros volume |


## 3. Demo executable — FastAPI + sklearn

L'approche minimaliste : un endpoint POST qui prend des features JSON et renvoie une prediction.


In [ ]:
# pip install -q fastapi 'uvicorn[standard]' pydantic scikit-learn joblib
import joblib, tempfile, os
from pathlib import Path

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from contextlib import asynccontextmanager

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

# === 1. Entrainer + sauvegarder un modele ===
data = load_iris(as_frame=True)
X, y = data.data, data.target
model = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, y)

model_path = Path(tempfile.gettempdir()) / "iris_rf.joblib"
joblib.dump({"model": model, "feature_names": list(X.columns), "target_names": list(data.target_names)}, model_path)
print(f"Modele sauve : {model_path}")

In [ ]:
# === 2. App FastAPI avec lifespan (load model once at startup) ===
ml_state: dict = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    print("Loading model...")
    ml_state.update(joblib.load(model_path))
    yield
    ml_state.clear()

app = FastAPI(title="Iris Classifier", version="1.0.0", lifespan=lifespan)

class Features(BaseModel):
    sepal_length: float = Field(..., ge=0, le=15)
    sepal_width: float = Field(..., ge=0, le=10)
    petal_length: float = Field(..., ge=0, le=15)
    petal_width: float = Field(..., ge=0, le=10)

class Prediction(BaseModel):
    class_id: int
    class_name: str
    probabilities: dict[str, float]

@app.get("/health")
def health():
    return {"status": "ready" if "model" in ml_state else "loading", "model": "iris_rf"}

@app.post("/predict", response_model=Prediction)
def predict(features: Features):
    import numpy as np
    X = np.array([[features.sepal_length, features.sepal_width,
                    features.petal_length, features.petal_width]])
    model = ml_state["model"]
    proba = model.predict_proba(X)[0]
    class_id = int(np.argmax(proba))
    return Prediction(
        class_id=class_id,
        class_name=ml_state["target_names"][class_id],
        probabilities={name: float(p) for name, p in zip(ml_state["target_names"], proba)},
    )

# === 3. Tester en in-process (sans demarrer un serveur reel) ===
with TestClient(app) as client:
    print(f"GET /health : {client.get('/health').json()}")
    r = client.post("/predict", json={"sepal_length": 5.1, "sepal_width": 3.5,
                                      "petal_length": 1.4, "petal_width": 0.2})
    print(f"POST /predict (setosa-like) : {r.json()}")
    r = client.post("/predict", json={"sepal_length": 7.0, "sepal_width": 3.2,
                                      "petal_length": 4.7, "petal_width": 1.4})
    print(f"POST /predict (versicolor-like) : {r.json()}")

## 4. BentoML — pattern recommande 2025

**BentoML** packe le modele + l'API en un "Bento" deployable sur K8s. Avantages : separation modele/runtime, gestion de versions, packaging propre.

```python
# pip install -q bentoml

import bentoml
from bentoml.io import NumpyNdarray, JSON
from pydantic import BaseModel

# 1. Sauvegarder dans le bentoml store (catalogue local)
bentoml.sklearn.save_model("iris_clf", model)

# 2. Definir le service
class Features(BaseModel):
    x: list[float]

clf_runner = bentoml.sklearn.get("iris_clf:latest").to_runner()
svc = bentoml.Service("iris_classifier", runners=[clf_runner])

@svc.api(input=JSON(pydantic_model=Features), output=JSON())
async def classify(features: Features):
    pred = await clf_runner.predict.async_run([features.x])
    return {"class_id": int(pred[0])}

# 3. Dev : bentoml serve service:svc --reload
# 4. Build : bentoml build
# 5. Containerize : bentoml containerize iris_classifier:latest
# 6. Deploy K8s / cloud
```

### Bento dynamic batching
```python
@bentoml.batchable(max_batch_size=64, max_latency_ms=10)
async def predict(input_batch):
    return model.predict(input_batch)
```
→ Agrege les requetes < 10ms en un seul batch GPU. Permet un **gain 5-20x** de throughput sur GPU sature.


## 5. TorchServe — PyTorch en prod

Standard pour servir des modeles PyTorch.

```bash
pip install torchserve torch-model-archiver

# 1. Archiver le modele
torch-model-archiver \
  --model-name resnet \
  --version 1.0 \
  --model-file model.py \
  --serialized-file resnet.pth \
  --handler image_classifier \
  --extra-files index_to_name.json

# 2. Lancer
torchserve --start --model-store ./store --models resnet=resnet.mar

# 3. Inference
curl http://127.0.0.1:8080/predictions/resnet -T image.jpg

# 4. Management : scale workers a la volee
curl -X PUT "http://127.0.0.1:8081/models/resnet?min_worker=4"
```

Features cles : **dynamic batching** natif, **model versioning**, **A/B test** integre, metrics Prometheus.


## 6. Triton Inference Server (NVIDIA)

Pour **multi-framework + GPU optimise**.

```
model_repository/
├── resnet_pytorch/
│   ├── config.pbtxt
│   └── 1/
│       └── model.pt
├── bert_onnx/
│   ├── config.pbtxt
│   └── 1/
│       └── model.onnx
```

```bash
docker run --gpus=all -p8000:8000 -p8001:8001 -p8002:8002 \
  -v $PWD/model_repository:/models nvcr.io/nvidia/tritonserver:24.05-py3 \
  tritonserver --model-repository=/models
```

Features killer :
- **Dynamic batching** (configurable par modele)
- **Multi-model**, multi-framework (PyTorch, TF, ONNX, TensorRT, OpenVINO, Python custom)
- **Ensemble models** : chaîner modeles dans Triton sans aller-retour Python
- **Sequence batching** pour stateful (LSTM)
- **Concurrent model execution** sur meme GPU


## 7. Ray Serve — Python distributed

Pour **scale Python natif** (au-dela d'un serveur).

```python
# pip install -q "ray[serve]"
from ray import serve
from starlette.requests import Request

@serve.deployment(num_replicas=4, ray_actor_options={"num_cpus": 2})
class Predictor:
    def __init__(self):
        import joblib
        self.model = joblib.load("model.pkl")

    async def __call__(self, request: Request) -> dict:
        data = await request.json()
        return {"prediction": int(self.model.predict([data["features"]])[0])}

serve.start()
serve.run(Predictor.bind())
```

Avantages : **pipelines** complexes (preprocess → predict → postprocess en parallele), **autoscaling** natif, integration Ray ecosystem (RLlib, Tune).


## 8. vLLM — LLM-specific (PagedAttention)

[vLLM](https://docs.vllm.ai/) = serveur d'inference avec **PagedAttention** (Kwon et al. 2023) → 24x plus de throughput que HF transformers naïf sur LLMs.

```bash
pip install vllm

# Mode programmatique
python -c "
from vllm import LLM, SamplingParams
llm = LLM(model='meta-llama/Meta-Llama-3-8B-Instruct', dtype='bfloat16')
outputs = llm.generate(['Hello'], SamplingParams(temperature=0.7, max_tokens=100))
print(outputs[0].outputs[0].text)
"

# Mode serveur (OpenAI-compatible)
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Meta-Llama-3-8B-Instruct \
  --gpu-memory-utilization 0.9 \
  --port 8000
```

Tu peux alors utiliser `openai` Python client en pointant vers `localhost:8000/v1`.


## 9. Patterns production critiques

### 9.1 Dynamic batching
Agreger les requetes < N ms pour saturer le GPU.
```python
# BentoML
@bentoml.batchable(max_batch_size=64, max_latency_ms=10)
async def predict(input_batch):
    return model.predict(input_batch)
```

### 9.2 Caching
```python
from functools import lru_cache

@lru_cache(maxsize=10000)
def predict_cached(features_tuple):
    return model.predict([features_tuple])
```

### 9.3 A/B test / Canary
```python
import random

@serve.deployment
class Router:
    async def __call__(self, request):
        if random.random() < 0.1:           # 10% du traffic
            return await V2Predictor.remote(request)
        return await V1Predictor.remote(request)
```

### 9.4 Shadow mode (dark launch)
Tu deploies v2 mais ne sers que v1 a l'utilisateur. v2 recoit toutes les requetes en parallele → tu compares offline.

### 9.5 Health checks
- **Readiness probe** : `GET /healthz` → 200 si modele charge
- **Liveness probe** : `GET /live` → 200 toujours sauf si deadlock
- K8s utilise ces probes pour decider quand router le trafic


## 10. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Charger le modele a chaque request | Lifespan + state |
| Pas de dynamic batching sur GPU | Saturation 10-20% au lieu de 90% |
| Endpoint `predict` sans validation | Pydantic obligatoire |
| Pas de versioning du modele | Model Registry (MLflow / BentoML Yatai) |
| Pas de health check | K8s ne sait pas si pret |
| Pas de metriques (Prometheus) | Latence / throughput / errors invisibles |
| Logs non structurees | JSON + request_id |
| Pas de rate limiting | DoS facile |
| Auth oubliee | API key minimum |
| Pas de strategy rollback | Blue/green ou canary |
| Pas de monitoring de drift | Cf. [MLOps_Drift_Monitoring.ipynb](./MLOps_Drift_Monitoring.ipynb) |

## 11. References
- **BentoML** : https://docs.bentoml.com/
- **TorchServe** : https://pytorch.org/serve/
- **Triton** : https://docs.nvidia.com/deeplearning/triton-inference-server/
- **Ray Serve** : https://docs.ray.io/en/latest/serve/
- **vLLM** : https://docs.vllm.ai/
- **KServe** : https://kserve.github.io/website/

Voir aussi :
- [FastAPI_API.ipynb](./FastAPI_API.ipynb) — base FastAPI
- [MLOps_Tracking_DVC_Wandb.ipynb](./MLOps_Tracking_DVC_Wandb.ipynb) — model registry
- [MLOps_Drift_Monitoring.ipynb](./MLOps_Drift_Monitoring.ipynb) — post-deploy monitoring
